# Reproducing *Learning to Solve QUBO in a Classification Way* (VCM)



This notebook uses hidden size h=4, depth d=15, instance size 4-5.

In [1]:
import itertools
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(5)
np.random.seed(5)
torch.set_printoptions(precision=3, sci_mode=False)

print("torch", torch.__version__)

torch 2.11.0+cpu


## 1. The QUBO objective

QUBO maximization quadratic form:

$$f(x) = x^{\top} Q x = \sum_i \sum_j q_{ij} x_i x_j, \qquad x \in \{0,1\}^n$$

Each variable set to 1 contributes its diagonal entry $q_{ii}$; each **pair** of variables both set
to 1 contributes $2q_{ij}$, because $Q$ is symmetric.

The 4-variable instance from the paper's Appendix C is used.

In [2]:
Q4 = torch.tensor([[ 49., -35.,  36., -89.],
                   [-35.,  42., -68.,  38.],
                   [ 36., -68., -30.,  92.],
                   [-89.,  38.,  92.,  24.]])

def ofv(Q, x):
    """Objective function value f(x) = x^T Q x. Supports a batch dimension."""
    return (x.unsqueeze(-2) @ Q @ x.unsqueeze(-1)).squeeze(-1).squeeze(-1)

def brute_force(Q):
    """Exhaustive search over all 2^n assignments. Only feasible for tiny n."""
    n = Q.shape[-1]
    best = None
    for bits in itertools.product([0., 1.], repeat=n):
        x = torch.tensor(bits)
        v = ofv(Q, x).item()
        if best is None or v > best[1]:
            best = ([int(b) for b in bits], v)
    return best

x_opt, f_opt = brute_force(Q4)
print("optimal assignment :", x_opt)
print("optimal objective  :", f_opt)

optimal assignment : [0, 0, 1, 1]
optimal objective  : 178.0


Checking the optimum by hand:
For $x = (0,0,1,1)$:

$$f = q_{33} + q_{44} + 2q_{34} = -30 + 24 + 2(92) = 178$$

Variable $x_3$ is **bad on its own** (its diagonal is $-30$), yet its $+92$ interaction with $x_4$,
counted twice, more than compensates. Good solutions therefore cannot be found by ranking variables
individually.

In [3]:
x = torch.tensor([0., 0., 1., 1.])
manual = Q4[2,2] + Q4[3,3] + 2*Q4[2,3]
print(f"f(0,0,1,1) via matrix form : {ofv(Q4, x).item():.0f}")
print(f"f(0,0,1,1) by hand         : {manual.item():.0f}")
print(f"x3 alone (diagonal)        : {Q4[2,2].item():.0f}   <- negative, yet x3 is in the optimum")

f(0,0,1,1) via matrix form : 178
f(0,0,1,1) by hand         : 178
x3 alone (diagonal)        : -30   <- negative, yet x3 is in the optimum


## 2. The objective-function increment (OFI)

Every prior learning-based solver flips one variable at a time, so it needs to know how much the
objective changes for each candidate flip. With $Q_k^{sum} = \sum_j q_{kj}x_j$ (how strongly
variable $k$ interacts with the currently active set), the paper gives the closed form

$$\mathrm{OFI}(x_k) = \begin{cases} q_{kk} + 2Q_k^{sum} & x_k = 0 \\ q_{kk} - 2Q_k^{sum} & x_k = 1\end{cases}$$

The authors then vectorise this into a single matrix expression covering all variables (and all
instances in a batch) at once — the change responsible for the roughly 1000x speed-up in their
Table 2, and what makes greedy repair cheap enough to run inside the training loop:

$$\mathrm{OFI}(x) = Q^{diag} + 2(1-2x)\,(Qx)$$

We implement both and assert that they agree.

In [4]:
def ofi_scalar(Q, x, k):
    """Closed form for a single variable k (paper, Appendix B)."""
    q_sum = (Q[k] * x).sum()
    return Q[k, k] + 2*q_sum if x[k] == 0 else Q[k, k] - 2*q_sum

def ofi_batch(Q, x):
    """Batched form: all variables (and instances) at once."""
    diag = torch.diagonal(Q, dim1=-2, dim2=-1)
    return diag + 2*(1 - 2*x)*(Q @ x.unsqueeze(-1)).squeeze(-1)

for probe in [torch.zeros(4), torch.tensor([1., 0., 1., 0.])]:
    s = torch.tensor([ofi_scalar(Q4, probe, k) for k in range(4)])
    b = ofi_batch(Q4, probe)
    assert torch.allclose(s, b), "scalar and batched OFI disagree"
    print(f"x = {[int(v) for v in probe.tolist()]}  ->  OFI = {[round(v,1) for v in b.tolist()]}")

print("\nBoth formulations agree.")

x = [0, 0, 0, 0]  ->  OFI = [49.0, 42.0, -30.0, 24.0]
x = [1, 0, 1, 0]  ->  OFI = [-121.0, -164.0, -42.0, 30.0]

Both formulations agree.


## 3. Greedy Flip (BGF) — reproducing the paper's walkthrough

Greedy Flip repeatedly applies the flip with the highest **positive** OFI and stops when no flip
improves the objective. Running it from the all-zero start reproduces the paper's Appendix C trace
exactly.

Note step 4: $x_1$ — greedy's very first pick — is switched back **off** once $x_4$ is active,
because $x_1$'s $-89$ conflict with $x_4$ now outweighs its own value. Local search must be able to
undo its earlier decisions.

In [5]:
def greedy_flip_trace(Q, x=None):
    """Greedy Flip from x (default all-zeros), returning the full trace."""
    n = Q.shape[-1]
    x = torch.zeros(n) if x is None else x.clone()
    trace = []
    while True:
        o = ofi_batch(Q, x)
        m, idx = o.max(-1)
        trace.append({
            "x": [int(v) for v in x.tolist()],
            "OFV": ofv(Q, x).item(),
            "OFI": [round(v, 1) for v in o.tolist()],
            "flip": int(idx) if m.item() > 0 else None,
        })
        if m.item() <= 0:
            break
        x[idx] = 1 - x[idx]
    return x, trace

x_g, trace = greedy_flip_trace(Q4)

print(f"{'x':<14}{'OFV':>7}   {'OFI per variable':<34}next")
print("-"*70)
for step in trace:
    nxt = f"flip x{step['flip']+1}" if step["flip"] is not None else "stop (no positive OFI)"
    print(f"{str(step['x']):<14}{step['OFV']:>7.0f}   {str(step['OFI']):<34}{nxt}")

print(f"\ngreedy result {[int(v) for v in x_g.tolist()]} with OFV {ofv(Q4, x_g).item():.0f}"
      f"  |  brute-force optimum {x_opt} with {f_opt:.0f}")

x                 OFV   OFI per variable                  next
----------------------------------------------------------------------
[0, 0, 0, 0]        0   [49.0, 42.0, -30.0, 24.0]         flip x1
[1, 0, 0, 0]       49   [-49.0, -28.0, 42.0, -154.0]      flip x3
[1, 0, 1, 0]       91   [-121.0, -164.0, -42.0, 30.0]     flip x4
[1, 0, 1, 1]      121   [57.0, -88.0, -226.0, -30.0]      flip x1
[0, 0, 1, 1]      178   [-57.0, -18.0, -154.0, -208.0]    stop (no positive OFI)

greedy result [0, 0, 1, 1] with OFV 178  |  brute-force optimum [0, 0, 1, 1] with 178


In [6]:
def bgf(Q, x, max_iter=200):
    """Batched Greedy Flip: repairs a batch of solutions to local optimality."""
    x = x.clone()
    single = (x.dim() == 1)
    if single:
        Q, x = Q.unsqueeze(0), x.unsqueeze(0)
    for _ in range(max_iter):
        o = ofi_batch(Q, x)
        m, idx = o.max(-1)
        active = m > 0
        if not active.any():
            break
        rows = torch.nonzero(active).squeeze(-1)
        x[rows, idx[rows]] = 1 - x[rows, idx[rows]]
    return x.squeeze(0) if single else x

# BGF never decreases the objective - the property GST relies on
Qb = torch.stack([Q4, Q4])
x_start = torch.tensor([[1., 1., 0., 0.], [0., 1., 0., 1.]])
x_rep = bgf(Qb, x_start)
for a, b in zip(x_start, x_rep):
    print(f"{[int(v) for v in a.tolist()]} (OFV {ofv(Q4,a).item():>5.0f})"
          f"  ->  {[int(v) for v in b.tolist()]} (OFV {ofv(Q4,b).item():>5.0f})")

[1, 1, 0, 0] (OFV    21)  ->  [0, 0, 1, 1] (OFV   178)
[0, 1, 0, 1] (OFV   142)  ->  [0, 0, 1, 1] (OFV   178)


## 4. The VCM model

Three components, following Equations (3)–(7) of the paper.

**Input scaling.** $\tanh$ saturates beyond an input magnitude of about 2, so raw row sums of $Q$
(which grow with $n$) would be squashed into indistinguishable values. Scaling by
$\lambda = \alpha/(s + s_B)$ places the first convolution in the sensitive region of $\tanh$
*independently of instance size or distribution* — the mechanism behind the paper's generalisation
claims.

**DVN (extractor).** Variable $x_j$ occupies both row $j$ and column $j$ of $Q$, and $Q = Q^{\top}$,
so its row-view and column-view must agree. The row-view of all variables is exactly $QV$, so DVN
reconciles $V$ with $QV$ by iterating

$$V^{(d),D} = M_3\big[\,V^{(d)}\;;\;\tanh(M_2\,\mathrm{ReLU}(M_1 QV^{(d)}))\,\big], \qquad V^{(d+1)} = \tanh(V^{(d),D})$$

Unlike a GCN there is **no degree normalisation** (the edge magnitudes *are* the objective) and the
weights $M_1, M_2, M_3$ are **shared across all depths** — DVN is a learned fixed-point iteration
rather than a deep network, which is what later permits extending depth at test time.

**VCN (classifier).** A projection to one score per variable, then a sign:

$$\mathrm{state}_i = \tanh(u \cdot \tanh(M_4 V^{(d),D})), \qquad x_i = \mathbb{1}[\mathrm{state}_i > 0]$$

All variables are classified simultaneously, so the $O(k\,n^2)$ of a sequential MDP becomes $O(n^2)$.

> **Implementation note.** We add a light residual damping ($0.85\,V + 0.4\,M_3[\cdot]$) to the DVN
> update. At $h=4$ the undamped iteration frequently fails to settle to a fixed point; damping makes
> the depth behaviour legible at this scale. This is a deviation from the paper and is revisited in
> Section 8.

In [7]:
class VCM(torch.nn.Module):
    """Miniature Value Classification Model: DVN extractor + VCN classifier."""

    def __init__(self, h=4, alpha=4.0, damping=(0.85, 0.4)):
        super().__init__()
        self.h, self.alpha, self.damping = h, alpha, damping
        self.M1 = torch.nn.Parameter(torch.randn(h, h)*0.3)
        self.M2 = torch.nn.Parameter(torch.randn(h, h)*0.3)
        self.M3 = torch.nn.Parameter(torch.randn(h, 2*h)*0.3)
        self.M4 = torch.nn.Parameter(torch.randn(h, h)*0.3)
        self.u  = torch.nn.Parameter(torch.randn(1, h)*0.3)

    def scale(self, Q):
        """lambda * Q, keeping the first convolution inside tanh's sensitive range."""
        s = Q.sum(-1).abs().max(-1).values
        lam = self.alpha / (s + s.mean() + 1e-6)
        return Q * lam.unsqueeze(-1).unsqueeze(-1)

    def dvn(self, Qs, depth, snapshots=False):
        """Depth Value Network: iterate the shared update, V^0 = ones."""
        a, b = self.damping
        V = torch.ones(*Qs.shape[:-2], self.h, Qs.shape[-1])
        snaps = []
        for _ in range(depth):
            C = V @ Qs                                        # graph convolution QV
            F = torch.tanh(self.M2 @ torch.relu(self.M1 @ C))  # filter, then compress
            V = torch.tanh(a*V + b*(self.M3 @ torch.cat([V, F], -2)))
            if snapshots:
                snaps.append(V.clone())
        return V, snaps

    def vcn(self, V):
        """Value Classification Network: one score per variable."""
        return torch.tanh(self.u @ torch.tanh(self.M4 @ V)).squeeze(-2)

    def forward(self, Q, depth=15, snapshots=False):
        V, snaps = self.dvn(self.scale(Q), depth, snapshots)
        return self.vcn(V), snaps

    @staticmethod
    def decide(state):
        return (state > 0).float()

model = VCM(h=4)
print(f"parameters: {sum(p.numel() for p in model.parameters())}  (paper uses h=128)")

parameters: 84  (paper uses h=128)


## 5. GST — greedy-guided self-training

A classifier needs labels, but optimal QUBO solutions are exactly the NP-hard object we cannot
compute. GST resolves this circularity per batch:

1. the model produces a solution;
2. **BGF repairs it** — guaranteed no worse, since every applied flip has positive OFI;
3. the repaired solution is compared against the **best ever seen** for that instance, and the
   winner becomes the label;
4. a binary cross-entropy loss on $(\mathrm{state}+1)/2$ pulls the classifier toward that label.

The model therefore trains on its own greedily-repaired outputs — self-distillation with a greedy
teacher — learning to emit directly what previously required a forward pass *plus* search.

We train on random Max-Cut QUBO instances (a ring guarantees connectivity, so no isolated node can
produce a zero row sum and a degenerate scaling factor).

In [8]:
def rand_maxcut_qubo(B, n, p=0.6):
    """Random weighted graphs -> Max-Cut QUBO matrices."""
    w = (torch.rand(B, n, n) < p).float() * torch.randint(1, 6, (B, n, n)).float()
    w = torch.triu(w, 1)
    ring = torch.zeros(B, n, n)
    for k in range(n):
        ring[:, k, (k+1) % n] = torch.randint(1, 4, (B,)).float()
    w = torch.triu(w + ring, 1)
    w = w + w.transpose(-1, -2)
    Q = -w
    i = torch.arange(n)
    Q[:, i, i] = w.sum(-1)
    return Q


def train_gst(model, n=5, steps=2600, batch=128, depth=15, lr=8e-3, verbose=True):
    """Greedy-guided Self Trainer (paper, Algorithm 1), without the historical-best store."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for t in range(steps):
        Q = rand_maxcut_qubo(batch, n)
        state, _ = model(Q, depth)
        x_vcm = VCM.decide(state).detach()
        label = bgf(Q, x_vcm)                       # greedy repair -> label
        p = ((state + 1)/2).clamp(1e-4, 1 - 1e-4)   # state -> probability
        loss = -(label*torch.log(p) + (1-label)*torch.log(1-p)).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if t % 200 == 0:
            history.append((t, loss.item()))
            if verbose:
                print(f"  step {t:>5}   BCE {loss.item():.4f}")
    return history

RESTARTS = [0, 1, 2, 3]
STEPS = 1600

print("training", len(RESTARTS), "restarts (about 30 s each on CPU)...")
runs = []
for seed in RESTARTS:
    torch.manual_seed(seed); np.random.seed(seed)
    m = VCM(h=4)
    train_gst(m, n=5, steps=STEPS, verbose=False)
    runs.append({"seed": seed, "model": m})
    print(f"  seed {seed}: done")

training 4 restarts (about 30 s each on CPU)...
  seed 0: done
  seed 1: done
  seed 2: done
  seed 3: done


Two deviations from the published procedure, both disclosed here and revisited in Section 8:

* The training set is regenerated every step, so the **historical-best store** of the full GST is
  omitted (there is no fixed instance whose best solution could be remembered). The paper's Figure 5
  shows that store is what stabilises label quality across epochs.
* At $h = 4$ the outcome varies noticeably with initialisation, so we train **several restarts** and
  select among them below. This mirrors the paper at a much smaller scale — the authors build their
  reference solution `VCM-BGF-HB` from 400 independently trained models.

## 6. Experiment A — Max-Cut on a concrete graph

Max-Cut asks for a partition of the vertices maximising the total weight of edges crossing the cut.
The standard reduction puts the **weighted degree on the diagonal** and the **negated edge weight**
off-diagonal:

$$q_{ii} = \sum_j w_{ij}, \qquad q_{ij} = -w_{ij}$$

This works because $x^{\top}Qx = \sum_{ij} w_{ij}(x_i + x_j - 2x_ix_j)$, and the bracket equals 1
exactly when the endpoints lie on opposite sides. Maximising $x^{\top}Qx$ therefore maximises the cut.

In [9]:
EDGES = [(0,1,3), (1,2,2), (2,3,4), (3,4,2), (4,0,2), (0,2,1), (1,3,2)]
N_NODES = 5

def graph_to_qubo(edges, n):
    W = torch.zeros(n, n)
    for i, j, w in edges:
        W[i, j] = w; W[j, i] = w
    Q = -W.clone()
    for i in range(n):
        Q[i, i] = W[i].sum()
    return Q

def cut_value(edges, x):
    return sum(w for i, j, w in edges if x[i] != x[j])

Q_graph = graph_to_qubo(EDGES, N_NODES)
print("weighted degrees :", [int(Q_graph[i, i]) for i in range(N_NODES)])
print("\nQUBO matrix Q:\n", Q_graph.int().numpy())

x_bf, f_bf = brute_force(Q_graph)
print(f"\nbrute force  : partition {x_bf}, objective {f_bf:.0f}, cut {cut_value(EDGES, x_bf)}")

weighted degrees : [6, 7, 7, 8, 4]

QUBO matrix Q:
 [[ 6 -3 -1  0 -2]
 [-3  7 -2 -2  0]
 [-1 -2  7 -4  0]
 [ 0 -2 -4  8 -2]
 [-2  0  0 -2  4]]

brute force  : partition [0, 1, 1, 0, 1], objective 14, cut 14


In [10]:
def evaluate(m, Q, edges, depth=15):
    """Run VCM on one instance and also report the BGF-repaired solution."""
    state, snaps = m(Q, depth=depth, snapshots=True)
    x_raw = VCM.decide(state)
    x_rep = bgf(Q, x_raw)
    traj = np.array([[m.vcn(V).tolist()[i] for V in snaps] for i in range(Q.shape[-1])])
    return {
        "state": state,
        "x_vcm": [int(v) for v in x_raw.tolist()],
        "x_bgf": [int(v) for v in x_rep.tolist()],
        "cut_vcm": cut_value(edges, [int(v) for v in x_raw.tolist()]),
        "cut_bgf": cut_value(edges, [int(v) for v in x_rep.tolist()]),
        "traj": traj,
        "settled": float(traj[:, -5:].std(axis=1).max()),
    }

for r in runs:
    r.update(evaluate(r["model"], Q_graph, EDGES))

opt = int(f_bf)
print(f"{'seed':>5}{'VCM cut':>10}{'VCM+BGF':>10}{'settled':>10}")
print("-"*35)
for r in runs:
    print(f"{r['seed']:>5}{r['cut_vcm']:>10}{r['cut_bgf']:>10}{r['settled']:>10.3f}")
print(f"\noptimum (brute force) = {opt}")

 seed   VCM cut   VCM+BGF   settled
-----------------------------------
    0         7        14     0.012
    1        11        11     0.706
    2         0        14     0.050
    3         7        14     0.021

optimum (brute force) = 14


We take the restart with the highest objective on this instance — breaking ties toward the one
whose depth iteration has settled, so the visualisation in Section 7 is legible.

In [11]:
best = max(runs, key=lambda r: (r["cut_vcm"], -r["settled"]))
print(f"selected restart: seed {best['seed']}")

x_vcm, cut_vcm = best["x_vcm"], best["cut_vcm"]
print("VCN states :", [round(v, 3) for v in best["state"].tolist()])
print(f"VCM cut    : {cut_vcm}   (optimum {opt})")

side0 = [i+1 for i, v in enumerate(x_vcm) if v == 0]
side1 = [i+1 for i, v in enumerate(x_vcm) if v == 1]
print(f"partition  : {side0} | {side1}")
print("crossing   :", [(i+1, j+1, w) for i, j, w in EDGES if x_vcm[i] != x_vcm[j]])

if cut_vcm == opt:
    print("\nOptimal cut reached in a single forward pass - no search.")
else:
    print(f"\nBelow optimum; BGF repair gives {best['cut_bgf']}.")

selected restart: seed 1
VCN states : [-0.929, 0.555, -0.907, 0.447, 0.655]
VCM cut    : 11   (optimum 14)
partition  : [1, 3] | [2, 4, 5]
crossing   : [(1, 2, 3), (2, 3, 2), (3, 4, 4), (5, 1, 2)]

Below optimum; BGF repair gives 11.


## 7. Experiment B — reliability, and what BGF is for

The restarts trained in Section 5 give an honest picture of how often one-shot classification reaches
the optimum at this scale, and how much of the shortfall the greedy repair of Section 3 recovers.

In [12]:
raw = [r["cut_vcm"] for r in runs]
rep = [r["cut_bgf"] for r in runs]
gap = lambda c: 100*(opt - c)/opt

print(f"{'seed':>5}{'VCM':>7}{'gap %':>8}{'+BGF':>7}{'gap %':>8}{'settled':>10}")
print("-"*45)
for r in runs:
    print(f"{r['seed']:>5}{r['cut_vcm']:>7}{gap(r['cut_vcm']):>8.1f}"
          f"{r['cut_bgf']:>7}{gap(r['cut_bgf']):>8.1f}{r['settled']:>10.3f}")

print(f"\noptimum {opt}")
print(f"VCM alone : {sum(c == opt for c in raw)}/{len(raw)} optimal, "
      f"mean cut {np.mean(raw):.1f}, mean gap {np.mean([gap(c) for c in raw]):.1f} %")
print(f"VCM + BGF : {sum(c == opt for c in rep)}/{len(rep)} optimal, "
      f"mean cut {np.mean(rep):.1f}, mean gap {np.mean([gap(c) for c in rep]):.1f} %")

 seed    VCM   gap %   +BGF   gap %   settled
---------------------------------------------
    0      7    50.0     14     0.0     0.012
    1     11    21.4     11    21.4     0.706
    2      0   100.0     14     0.0     0.050
    3      7    50.0     14     0.0     0.021

optimum 14
VCM alone : 0/4 optimal, mean cut 6.2, mean gap 55.4 %
VCM + BGF : 3/4 optimal, mean cut 13.2, mean gap 5.4 %
